# 🚗 Lightning McQueen — StyleTTS2 Trainer (Definitive Kaggle T4 Edition)

**Verified for**: Kaggle T4 GPU · Python 3.12 · PyTorch 2.10.0+cu128

### Before clicking Run All:
1. Right sidebar → **Settings** → **Accelerator** → **GPU T4 x2**
2. Right sidebar → **Settings** → **Internet** → **ON**
3. Make sure your audio dataset is added under **Input** on the right sidebar
4. Update `INPUT_FILE` in **Cell 3** to match your wav file path shown in the Input panel
5. Click **Run All** — do not add any extra cells

## Cell 1 — Install Dependencies
⚠️ We NEVER install `torch` here. Kaggle pre-installs `2.10.0+cu128` which is perfectly compatible with T4 (sm_75). Installing torch from pip replaces it with a broken version.

In [ ]:
!apt-get update -qq
!apt-get install -y espeak-ng -qq
!git clone https://github.com/yl4579/StyleTTS2.git /kaggle/working/StyleTTS2

# Core deps — do NOT include torch/torchvision/torchaudio
!pip install -q pydub faster-whisper munch pyyaml librosa soundfile nltk tensorboard einops einops-exts tqdm phonemizer accelerate

# transformers pinned to 4.40.0 with --no-deps to prevent it from touching PyTorch
!pip install -q 'transformers==4.40.0' --no-deps
# Manually install transformers' required deps (none of these touch torch)
!pip install -q "tokenizers>=0.19,<0.20" huggingface-hub safetensors regex filelock packaging requests

# monotonic-alignment-search: native Python 3.12 support, no Cython required
!pip install -q monotonic-alignment-search

# Verify torch is still the correct GPU version
import torch
assert torch.cuda.is_available(), '❌ CUDA not available! Do NOT install torch from pip.'
assert 'cpu' not in torch.__version__, '❌ CPU-only torch detected! Factory reset and re-run.'
print(f'✅ PyTorch {torch.__version__} | GPU: {torch.cuda.get_device_name(0)} | VRAM: {torch.cuda.get_device_properties(0).total_memory // 1024**3}GB')

## Cell 2 — Apply All Patches
Fixes PyTorch 2.6+ `weights_only` security change, monotonic_align import rename, squeeze dimension bugs, HiFiGAN F0 crashes, and GPU memory leaks.

In [ ]:
# Fix 0: Restore losses.py just in case it was corrupted by previous failed patches
!curl -s https://raw.githubusercontent.com/yl4579/StyleTTS2/main/losses.py -o /kaggle/working/StyleTTS2/losses.py

import os, re
os.chdir('/kaggle/working/StyleTTS2')

def patch(path, replacements):
    with open(path, 'r', encoding='utf-8') as f:
        code = f.read()
    for old, new in replacements:
        code = old.sub(new, code) if isinstance(old, re.Pattern) else code.replace(old, new)
    with open(path, 'w', encoding='utf-8') as f:
        f.write(code)

# Fix 1: monotonic_alignment_search renamed the module — patch the import
patch('utils.py', [
    ('from monotonic_align import maximum_path',
     'from monotonic_alignment_search import maximum_path'),
])

# Fix 2: PyTorch 2.6+ made weights_only=True the default, breaking old .pth files
patch('models.py', [
    ("torch.load(model_path, map_location='cpu')",
     "torch.load(model_path, map_location='cpu', weights_only=False)"),
    ("torch.load(model_path)",
     "torch.load(model_path, weights_only=False)"),
    # Fix h.view crash when a batch of size 1 squeezed incorrectly
    (re.compile(r'([ \t]*)h = h\.view\(h\.size\(0\), h\.size\(1\), 1\)'),
     r'\1if h.ndim == 1:\n\1    h = h.unsqueeze(0)\n\1h = h.view(h.size(0), h.size(1), 1)')
])

patch('train_first.py', [
    ("torch.load(path, map_location='cpu')",
     "torch.load(path, map_location='cpu', weights_only=False)")
])

patch('train_second.py', [
    ("torch.load(path, map_location='cpu')",
     "torch.load(path, map_location='cpu', weights_only=False)"),
    # Fix squeeze() which can collapse batch dim — must specify dim explicitly
    ('s_dur = torch.stack(ss).squeeze()',
     's_dur = torch.stack(ss).squeeze(1)'),
    ('gs = torch.stack(gs).squeeze()',
     'gs = torch.stack(gs).squeeze(1)'),
    ('s = torch.stack(ss).squeeze()',
     's = torch.stack(ss).squeeze(1)'),
    ('loss_mel = stft_loss(y_rec.squeeze(), wav.detach())',
     'loss_mel = stft_loss(y_rec.squeeze(1), wav.detach().squeeze(1))'),
    # Free GPU memory after each epoch to prevent fragmentation
    ("print('Epochs: %d\\nValidation loss: %.3f, Dur loss: %.3f, F0 loss: %.3f' % (epoch, val_loss, val_dur_loss, val_f0_loss))",
     "print('Epochs: %d\\nValidation loss: %.3f, Dur loss: %.3f, F0 loss: %.3f' % (epoch, val_loss, val_dur_loss, val_f0_loss))\n        import gc; gc.collect(); torch.cuda.empty_cache()")
])

# Fix 3: HiFiGAN F0 dimension crashes
patch('Modules/hifigan.py', [
    (re.compile(r'([ \t]*)F0_curve = nn\.functional\.conv1d\(F0_curve\.unsqueeze\(1\)'),
     r'\1if F0_curve.ndim == 3 and F0_curve.shape[-1] == 1:\n\1    F0_curve = F0_curve.squeeze(-1)\n\1if F0_curve.ndim == 1:\n\1    F0_curve = F0_curve.unsqueeze(0)\n\1F0_curve = nn.functional.conv1d(F0_curve.unsqueeze(1)'),
    (re.compile(r'([ \t]*)F0 = self\.F0_conv\(F0_curve\.unsqueeze\(1\)\)'),
     r'\1if F0_curve.ndim == 1:\n\1    F0_curve = F0_curve.unsqueeze(0)\n\1F0 = self.F0_conv(F0_curve.unsqueeze(1))')
])

print('✅ All patches applied successfully!')

## Cell 3 — Slice Audio & Transcribe
⚠️ Update `INPUT_FILE` to match the path shown in your Kaggle **Input** panel on the right side.

In [ ]:
import os
from pydub import AudioSegment
from pydub.silence import split_on_silence
from faster_whisper import WhisperModel

VOICE_NAME = 'mcqueen'

# ⬇️ UPDATE THIS to the exact path shown in your Kaggle Input panel
INPUT_FILE = '/kaggle/input/datasets/infernapeshashank/asshole/Mcqueensample.wav'

OUTPUT_DIR = f'/kaggle/working/StyleTTS2/Data/{VOICE_NAME}_wavs'
os.makedirs(OUTPUT_DIR, exist_ok=True)

if not os.path.exists(INPUT_FILE):
    raise FileNotFoundError(
        f'Cannot find: {INPUT_FILE}\n'
        'Check the right sidebar Input panel and update INPUT_FILE above!'
    )

print('✂️  Slicing audio...')
audio = AudioSegment.from_file(INPUT_FILE)
chunks = split_on_silence(
    audio,
    min_silence_len=500,
    silence_thresh=audio.dBFS - 14,
    keep_silence=250
)
clip_paths = []
for idx, chunk in enumerate(chunks):
    if len(chunk) > 1000:
        out = os.path.join(OUTPUT_DIR, f'slice_{idx:04d}.wav')
        chunk.export(out, format='wav')
        clip_paths.append(out)
print(f'✅ Sliced into {len(clip_paths)} clips')

# Use CPU for Whisper — Kaggle's faster-whisper has CUDA driver version issues
print('📝 Transcribing with Whisper (CPU)...')
wmodel = WhisperModel('base.en', device='cpu', compute_type='int8')
items = []
for i, path in enumerate(clip_paths):
    segs, _ = wmodel.transcribe(path, beam_size=5)
    text = ' '.join([s.text for s in segs]).strip()
    if text:
        items.append({'audio': f'{VOICE_NAME}_wavs/' + os.path.basename(path), 'text': text})
        print(f'  [{i+1:02d}] {os.path.basename(path)}: "{text}"')

if len(items) < 2:
    raise ValueError('Need at least 2 clips! Try a longer audio file.')

split = max(1, int(len(items) * 0.95))
train_items = items[:split]
val_items = items[split:] if items[split:] else [items[-1]]

data_dir = f'/kaggle/working/StyleTTS2/Data/{VOICE_NAME}'
os.makedirs(data_dir, exist_ok=True)
with open(f'{data_dir}/train_list.txt', 'w', encoding='utf-8') as f:
    for item in train_items:
        f.write(f"{item['audio']}|{item['text']}|0\n")
with open(f'{data_dir}/val_list.txt', 'w', encoding='utf-8') as f:
    for item in val_items:
        f.write(f"{item['audio']}|{item['text']}|0\n")

print(f'✅ Dataset: {len(train_items)} train / {len(val_items)} val clips')

## Cell 4 — Download Models & Build Config

In [ ]:
import yaml, urllib.request, os
os.chdir('/kaggle/working/StyleTTS2')

def download(url, path):
    if not os.path.exists(path):
        print(f'  Downloading {os.path.basename(path)}...')
        urllib.request.urlretrieve(url, path)
    else:
        print(f'  {os.path.basename(path)} — already exists')

os.makedirs('Utils/ASR', exist_ok=True)
os.makedirs('Utils/JDC', exist_ok=True)
os.makedirs('Models/LibriTTS', exist_ok=True)
os.makedirs('Models/mcqueen', exist_ok=True)

print('📥 Downloading utility models...')
# ASR model (verified URL from working local setup)
download(
    'https://huggingface.co/yl4579/StyleTTS2-LibriTTS/resolve/main/Models/LibriTTS/epoch_00080.pth',
    'Utils/ASR/epoch_00080.pth'
)
# JDC pitch model (correct name: bst.t7, NOT bst.conformer.pth)
download(
    'https://github.com/nickoala/jdc/raw/master/bst.t7',
    'Utils/JDC/bst.t7'
)
# Base checkpoint for fine-tuning
download(
    'https://huggingface.co/yl4579/StyleTTS2-LibriTTS/resolve/main/Models/LibriTTS/epochs_2nd_00020.pth',
    'Models/LibriTTS/epochs_2nd_00020.pth'
)
print('✅ Models ready!')

print('⚙️  Building config...')
urllib.request.urlretrieve(
    'https://raw.githubusercontent.com/yl4579/StyleTTS2/main/Configs/config_ft.yml',
    'config_base.yml'
)
with open('config_base.yml', 'r') as f:
    cfg = yaml.safe_load(f)

# Data paths
cfg['data_params']['root_path'] = 'Data'          # folder containing wav subfolders
cfg['data_params']['train_data'] = 'Data/mcqueen/train_list.txt'
cfg['data_params']['val_data']   = 'Data/mcqueen/val_list.txt'

# Training settings
cfg['log_dir']          = 'Models/mcqueen/'
cfg['batch_size']       = 2      # MINIMUM 2 required for WavLM loss to work
cfg['max_len']          = 128    # Safe VRAM usage on single T4 (15GB)
cfg['pretrained_model'] = 'Models/LibriTTS/epochs_2nd_00020.pth'
cfg['max_epoch']        = 250
cfg['save_freq']        = 25

if 'loss_params' not in cfg:
    cfg['loss_params'] = {}
cfg['loss_params']['TMA_epoch'] = 50

with open('config_stage2.yml', 'w') as f:
    yaml.dump(cfg, f)
print('✅ Config ready — 250 epochs, batch_size=2, max_len=128')

## Cell 5 — 🚀 Launch Training
`CUDA_VISIBLE_DEVICES=0` forces single-GPU mode to prevent DataParallel from splitting tensors across T4 x2 (which crashes with batch_size<4).

Expected runtime: ~2–3 hours for 250 epochs.

In [ ]:
import os
os.chdir('/kaggle/working/StyleTTS2')
# CUDA_VISIBLE_DEVICES=0   → use only GPU 0, disables DataParallel
# expandable_segments=True → reduces memory fragmentation, prevents OOM
!CUDA_VISIBLE_DEVICES=0 PYTORCH_CUDA_ALLOC_CONF=expandable_segments:True python train_second.py --config_path ./config_stage2.yml

## Cell 6 — 🎉 Export Final Model
Run this after training finishes to copy the final checkpoint to a downloadable location.

In [ ]:
import shutil, glob, os

ckpts = sorted(glob.glob('/kaggle/working/StyleTTS2/Models/mcqueen/epoch_2nd_*.pth'))
if ckpts:
    latest = ckpts[-1]
    dest = '/kaggle/working/mcqueen_final.pth'
    shutil.copy(latest, dest)
    size_mb = os.path.getsize(dest) / 1024 / 1024
    print(f'🎉 Model saved: {dest}  ({size_mb:.0f} MB)')
    print(f'   Source: {os.path.basename(latest)}')
    print('   Download it from the Output panel on the right sidebar!')
else:
    print('❌ No checkpoint found. Did training finish?')
    print('   Check for epoch_2nd_*.pth in /kaggle/working/StyleTTS2/Models/mcqueen/')